# Matched Qwen-7B BS Activation Patching

This notebook uses the shared `activation_patching.py` runner to select 100 matched BS examples for `DeepSeek-R1-Distill-Qwen-7B`, cache that selection, and run two last-token residual patching experiments:

- Denoising: deceptive prefix patched with the matched truthful prefix.
- Noising: truthful prefix patched with the matched deceptive prefix.

The pair cache lives under `deception2/Cache/activation_patching`; generation outputs stream to JSONL and live summary CSVs under `deception2/Results/activation_patching`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.options.display.max_colwidth = 220
pd.options.display.max_columns = 200

REPO_ROOT = Path('/playpen-ssd/smerrill/deception2')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from activation_patching import (
    DEFAULT_LOCALIZATION_DIR,
    DEFAULT_MODEL_NAME,
    DEFAULT_PAIR_CACHE_PATH,
    build_default_layer_candidates,
    load_or_build_bs_activation_patch_pair_cache,
    parse_layer_candidates,
    run_matched_pair_patch_experiment,
)


def md(text: str) -> None:
    display(Markdown(text))

In [ ]:
MODEL_NAME_OR_PATH = os.environ.get('ACT_PATCH_MODEL_NAME', DEFAULT_MODEL_NAME)
LOCALIZATION_DIR = Path(os.environ.get('ACT_PATCH_LOCALIZATION_DIR', str(DEFAULT_LOCALIZATION_DIR)))
PAIR_CACHE_PATH = Path(os.environ.get('ACT_PATCH_PAIR_CACHE_PATH', str(DEFAULT_PAIR_CACHE_PATH)))
OUTPUT_ROOT = REPO_ROOT / 'Results' / 'activation_patching'
RUN_TAG = os.environ.get('ACT_PATCH_RUN_TAG', 'bs_DeepSeek-R1-Distill-Qwen-7B_matched100_last_token_residual')
RUN_DIR = OUTPUT_ROOT / RUN_TAG

PAIR_COUNT = int(os.environ.get('ACT_PATCH_PAIR_COUNT', '100'))
REFRESH_PAIR_CACHE = os.environ.get('ACT_PATCH_REFRESH_PAIR_CACHE', '0') == '1'

PATCH_MAX_MODEL_LENGTH = int(os.environ.get('ACT_PATCH_MAX_MODEL_LENGTH', '10000'))
PATCH_MAX_NEW_TOKENS = int(os.environ.get('ACT_PATCH_MAX_NEW_TOKENS', '10000'))
PATCH_RATE_SAMPLE_COUNT = int(os.environ.get('ACT_PATCH_RATE_SAMPLE_COUNT', '50'))
PATCH_TEMPERATURE = float(os.environ.get('ACT_PATCH_TEMPERATURE', '0.8'))
PATCH_TOP_P = float(os.environ.get('ACT_PATCH_TOP_P', '0.95'))
PATCH_BASE_SEED = int(os.environ.get('ACT_PATCH_BASE_SEED', '17'))
PATCH_CUDA_DEVICE = os.environ.get('ACT_PATCH_CUDA_DEVICE', 'cuda:0')

# Defaults to 0,2,5,... after the model is loaded. Override with e.g. ACT_PATCH_LAYER_CANDIDATES=0,2,5.
MANUAL_LAYER_CANDIDATES = parse_layer_candidates(os.environ.get('ACT_PATCH_LAYER_CANDIDATES', ''))

# Leave off for preview/cache work; set ACT_PATCH_RUN_EXPERIMENT=1 or edit this to launch the GPU run.
RUN_EXPERIMENT = os.environ.get('ACT_PATCH_RUN_EXPERIMENT', '0') == '1'

assert LOCALIZATION_DIR.exists(), LOCALIZATION_DIR
assert PAIR_COUNT > 0
assert PATCH_RATE_SAMPLE_COUNT > 0
assert PATCH_MAX_MODEL_LENGTH > 0
assert PATCH_MAX_NEW_TOKENS > 0

In [ ]:
pairs_df = load_or_build_bs_activation_patch_pair_cache(
    LOCALIZATION_DIR,
    pair_cache_path=PAIR_CACHE_PATH,
    pair_count=PAIR_COUNT,
    refresh_cache=REFRESH_PAIR_CACHE,
)

md('## Cached Matched Pairs')
md(
    "\n".join(
        [
            f'- Cache: `{PAIR_CACHE_PATH}`',
            f'- Pairs loaded: `{len(pairs_df)}`',
            f'- Median commitment delta: `{pairs_df["commitment_delta"].median():.3f}`',
            f'- Median deceptive-prefix rate: `{pairs_df["deceptive_prefix_deception_rate"].median():.3f}`',
            f'- Median donor clarity: `{pairs_df["donor_clarity_score"].median():.3f}`',
        ]
    )
)

display(
    pairs_df[
        [
            'pair_id',
            'example_id',
            'shared_context_sentence_pos',
            'commitment_sentence_pos',
            'shared_context_deception_rate',
            'deceptive_prefix_deception_rate',
            'commitment_delta',
            'donor_generation_idx',
            'donor_clarity_score',
            'n_truthful_donors',
            'deceptive_commitment_sentence',
            'truthful_donor_sentence',
        ]
    ].head(12)
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
pairs_df['commitment_delta'].plot.hist(ax=axes[0], bins=25, title='Commitment spike')
pairs_df['deceptive_prefix_deception_rate'].plot.hist(ax=axes[1], bins=25, title='Deceptive prefix rate')
pairs_df['donor_clarity_score'].plot.hist(ax=axes[2], bins=25, title='Donor clarity')
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
pair = pairs_df.iloc[0]
md('## Top Pair Preview')
md(
    "\n".join(
        [
            f'- Pair: `{pair["pair_id"]}`',
            f'- Example: `{pair["example_id"]}`',
            f'- Spike: `{pair["shared_context_deception_rate"]:.3f} -> {pair["deceptive_prefix_deception_rate"]:.3f}`',
            f'- Delta: `{pair["commitment_delta"]:.3f}`',
            f'- Truthful donor generation: `{int(pair["donor_generation_idx"])}`',
        ]
    )
)

md('### Shared Prefix Tail')
print(pair['shared_context_text'][-1500:])

md('### Deceptive Commitment Sentence')
print(pair['deceptive_commitment_sentence'])

md('### Truthful Donor Sentence')
print(pair['truthful_donor_sentence'])

md('### Saved Truthful Donor Rollout')
print(pair['donor_full_generation_text'])

In [ ]:
if RUN_EXPERIMENT:
    run_matched_pair_patch_experiment(
        pairs_df=pairs_df,
        output_root=RUN_DIR,
        model_name_or_path=MODEL_NAME_OR_PATH,
        max_model_length=PATCH_MAX_MODEL_LENGTH,
        max_new_tokens=PATCH_MAX_NEW_TOKENS,
        samples_per_condition=PATCH_RATE_SAMPLE_COUNT,
        temperature=PATCH_TEMPERATURE,
        top_p=PATCH_TOP_P,
        base_seed=PATCH_BASE_SEED,
        cuda_device_name=PATCH_CUDA_DEVICE,
        layer_candidates=MANUAL_LAYER_CANDIDATES,
        include_baselines=True,
        patch_scope='last_token',
        resume=True,
    )
else:
    md('## Experiment Not Launched')
    md(
        'Set `RUN_EXPERIMENT = True` in the config cell or run the notebook with '
        '`ACT_PATCH_RUN_EXPERIMENT=1` to launch the full GPU sweep.'
    )
    print('Script equivalent:')
    print(
        f"python {SRC_ROOT / 'activation_patching.py'} "
        f"--pair-count {PAIR_COUNT} "
        f"--pair-cache-path {PAIR_CACHE_PATH} "
        f"--output-root {OUTPUT_ROOT} "
        f"--run-tag {RUN_TAG} "
        f"--cuda-device {PATCH_CUDA_DEVICE}"
    )

In [ ]:
md('## Live Results')
summary_path = RUN_DIR / 'condition_summary_live.csv'
pair_summary_path = RUN_DIR / 'pair_condition_summary_live.csv'
samples_path = RUN_DIR / 'samples.jsonl'

print('Run dir:', RUN_DIR)
print('Samples JSONL:', samples_path, 'exists=', samples_path.exists())
print('Condition summary:', summary_path, 'exists=', summary_path.exists())

if summary_path.exists():
    condition_summary_df = pd.read_csv(summary_path)
    display(condition_summary_df)

if pair_summary_path.exists():
    pair_condition_summary_df = pd.read_csv(pair_summary_path)
    display(pair_condition_summary_df.head(20))

## Output Files

The runner writes these files continuously:

- `samples.jsonl`: one row per generated continuation, flushed immediately.
- `condition_summary_live.csv`: pooled baseline, denoising, and noising rates across completed pairs.
- `pair_condition_summary_live.csv`: per-pair rates for each condition.
- `matched_pairs.jsonl` / `matched_pairs.csv`: the cached matched pair metadata copied into the run directory.
- `token_debug_live.csv`: token counts and final-token previews for each pair as it is processed.